# Microsoft OneLake Hands-on Workshop

## Getting started with Fabric

#### Sign up for a free Fabric account - [Free account](https://www.microsoft.com/en-us/microsoft-fabric/getting-started)
#### Setup a new trial capacity - [Account manager](https://learn.microsoft.com/en-us/fabric/fundamentals/fabric-trial#method-1)

## Creating your Lakehouse and adding sample data

1. Login to Fabric and create a new Item in your default Workspace (named "My Workspace")

2. Select **Lakehouse** and name it *iceberglake*.  Ensure *Lakehouse Schemas* is checked.  Click **Create**.

3. Select **Start with sample data** 

4. Select *Retail data model from Wide World Importers*

After a few seconds, you should have a set of tables in your Lakehouse.


## Creating your Spark runtime environment

Since Iceberg isn't yet included in Fabric Spark (coming soon), you'll need to create a runtime environment and include the Iceberg libraries. Don't worry, it's easy.

First, download to your local machine [Iceberg 1.10.1 for Spark 3.5](https://search.maven.org/remotecontent?filepath=org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.10.1/iceberg-spark-runtime-3.5_2.12-1.10.1.jar) and [Azure-bundle 1.10.1](https://search.maven.org/remotecontent?filepath=org/apache/iceberg/iceberg-azure-bundle/1.10.1/iceberg-azure-bundle-1.10.1.jar). These are also listed on the [Iceberg Releases page](https://iceberg.apache.org/releases/).

Second, in your **My Workspace**, click **+New item** from the top-left corner. \
Select **Environment** (make it easier to find, type *environment* in the filter box on the top-right) \
Name it **iceberg_env_1.10.1**

Select **Custom** from the left-hand side \
Click **Upload** and upload both JAR files you downloaded from the Iceberg site. \
Click **Save** \
Click **Publish**

## Importing the workshop notebook

Download the notebook from [Gist](https://gist.github.com/rhasson/02872b4c906be1467a9b56b41b037d79) \
From your **My Workspace**, click **Import --> Notebook --> From this computer** \
Select the notebook you downloaded and import it.

## Setting up Spark session and catalogs

### Tale of two catalogs
Today, OneLake catalog only supports the ability to read Iceberg tables. Writing can be accomplished in two ways:
1. Create a Delta Lake table which is automatically converted to Iceberg by OneLake - the sample data you created previously
2. Use the Hadoop filesystem catalog to write Iceberg tables directly into OneLake.  Tables will be added automatically to the OneLake catalog by the virtualization layer (takes a few seconds)

You will configure the Spark environment with two catalogs. 

1. **readcat** - REST catalog pointing to the Iceberg REST APIs in OneLake
2. **writecat** - Hadoop catalog pointing to OneLake table path to store manifests and data files

In [ ]:
from pyspark.sql import SparkSession

# OneLake Table API endpoint
table_api_url = "https://onelake.table.fabric.microsoft.com/iceberg"

# Catalog auth via Entra token
token = notebookutils.credentials.getToken('storage')

workspace = "My Workspace"
lakehouse = "iceberglake.Lakehouse"

write_warehouse = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}/Tables"
read_warehouse = f"{workspace}/{lakehouse}"

# Start SparkSession with Iceberg REST Catalog support
# These are all standard OSS configurations
# https://iceberg.apache.org/docs/latest/spark-configuration/#catalog-configuration
spark = (
    SparkSession.builder
    .appName("Iceberg OneLake Workshop")
    .config(
        'spark.jars.packages',
        'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.1,'
        'org.apache.iceberg:iceberg-azure-bundle:1.10.1,'
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.readcat", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.readcat.type", "rest")
    .config("spark.sql.catalog.readcat.uri", table_api_url)
    .config("spark.sql.catalog.readcat.token", token)
    .config("spark.sql.catalog.readcat.warehouse", read_warehouse)
    .config("spark.sql.catalog.readcat.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    .config("spark.sql.catalog.writecat", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.writecat.type", "hadoop")
    .config("spark.sql.catalog.writecat.warehouse", write_warehouse)
    .config("spark.sql.catalog.writecat.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    .config("spark.sql.defaultCatalog", "readcat")

    .getOrCreate()
)

# confirm required Iceberg jars are loaded
sc = spark.sparkContext
print(sc._jsc.sc().listJars())

## Exploring using Iceberg REST APIs in OneLake

In the next few cells you'll explore the lakehouse you created with sample data. \
You'll use the *readcat* catalog which utilized the OneLake Table APIs to retrieve Iceberg table metadata - schema, location, etc. \
Through this catalog, engines, like Fabric Spark, can query Iceberg tables with ease.

First, lets see which tables are available.  If you follow the setup steps above, you should have a few tables with sample data in your lakehouse.

In [ ]:
%%sql

show tables in readcat.dbo

Second, lets query one of these.  I'll choose the fact table.  If you loaded different sample datasets, you will need to update the table name below.

In [ ]:
%%sql

select * from readcat.dbo.fact_sale limit 10

Describe the fact table to understand the schema, data types, etc.

In [ ]:
%%sql

describe formatted readcat.dbo.fact_sale

Next, lets get a little more creative and join the fact table with the Country dimension table to get a more concise view.

In [ ]:
%%sql

select s.Profit, s.Description, c.Country 
from readcat.dbo.fact_sale as s 
join readcat.dbo.dimension_city as c on s.CityKey = c.CityKey 
limit 10

## Creating Iceberg tables in OneLake

In this section you'll learn how to create Iceberg tables directly in OneLake.

Today, the OneLake REST APIs don't yet support write capabilities, so we'll use the *writecat* which utilizes Hadoop/filesystem catalog to write. \
Doing this will write Iceberg tables into your Lakehouse's */Tables* folder. These tables will be automatically added to the catalog. They will also be converted to Delta Lake.

### Creating sample Iceberg table

The following cell downloads NYC Taxi dataset, extracts only the station information (where bikes are parked) and writes it into a OneLake.

**BONUS:** Explore the [NYC Open Data](https://opendata.cityofnewyork.us/) page and bring in other interesting datasets to use in this workshop.

Notice that we're using the *writecat* catalog here

First, create a new schema for you to use.  This is just so we can easily create, drop and manipulate tables without impacting other things.

In [ ]:
%%sql

create schema writecat.test

Next, run the following cell to download the bike station data and create the **stations** table in the **test** schema you just created

In [ ]:
import requests
import json
import pandas as pd

### https://data.cityofnewyork.us/NYC-BigApps/Citi-Bike-System-Data/vsnr-94wk
r = requests.get('https://gbfs.citibikenyc.com/gbfs/en/station_status.json')
station_status = r.json()
output = []

for item in station_status['data']['stations']:
  output.append(item)

pdf = pd.DataFrame(output)
df_stations = spark.createDataFrame(pdf)

## write sample dataset into OneLake as an Iceberg table using Hadoop catalog
df_stations.createOrReplaceTempView('stations')
spark.sql('CREATE OR REPLACE TABLE writecat.test.stations USING iceberg AS SELECT * FROM stations')

Check that your table has been created.

In [ ]:
%%sql

show tables in readcat.test

In [ ]:
%%sql 

select * from readcat.test.stations limit 10

### Manipulating Iceberg Tables

Creating an Iceberg table from raw data is pretty simple. You can use this approach for your own ETL jobs.

Next, lets create another table that we'll use to manually insert, update and delete rows. \
If you feel ambitious, add a few more columns to the table definition.  Later, make sure to populate them with the INSERT statement

In [ ]:
%%sql

CREATE OR REPLACE TABLE  writecat.test.users (
    id INT,
    name STRING
) using iceberg;

We now have an empty Iceberg table.  Lets add a few rows.

In [ ]:
%%sql

INSERT INTO writecat.test.users VALUES (1, 'Alice'), (2, 'Bob'), (3, 'Charlie');

In [ ]:
%%sql

SELECT * FROM readcat.test.users;

Now you should have a simple table with a few rows. The above SELECT statement should have returned those results.

Next, lets update a single row. This demonstrates how easy it is to update your data without any custom ETL or fancy code.

In [ ]:
%%sql

UPDATE writecat.test.users SET name = 'Bobby' WHERE id = 2; 

In [ ]:
%%sql

SELECT * FROM readcat.test.users;

Nice!  You should see the second record was changed from Bob to Bobby

Next, try deleting a row from the table.  The cell below is a hint, you'll need to complete it before it will execute correctly

In [ ]:
%%sql

DELETE FROM writecat.test.users [WHERE predicate]

In [ ]:
%%sql 

SELECT * FROM readcat.test.users;

## Using 3rd Party engines

Since the data in OneLake is stored in Iceberg format and access is available using Iceberg REST APIs \
all other compatible (understand Iceberg, uses Iceberg REST catalog and can read from OneLake) can easily \
discover and query these tables.

**Lets pause for a quick demo of using Snowflake and Clickhouse with Iceberg tables in OneLake**

Now you can try this on your own.  One easy way is to load DuckDB (in-process OLAP engine) in this notebooks \
and run a few queries against your Iceberg tables.

In [ ]:
%pip install duckdb -q

In [ ]:
import duckdb

con = duckdb.connect()

# Install & load extensions
con.execute("INSTALL iceberg; LOAD iceberg;")
con.execute("INSTALL azure; LOAD azure;")
con.execute("INSTALL httpfs; LOAD httpfs;")

token = notebookutils.credentials.getToken('storage')

Create the Secret objects that DuckDB will use to access the OneLAke catalog and storage

In [ ]:
# Secret for the Iceberg REST Catalog
con.execute("""
    CREATE OR REPLACE SECRET onelake_catalog_secret (
        TYPE ICEBERG,
        TOKEN ?
    );
""", [token])

In [ ]:
# Secret for OneLake filesystem
con.execute("""
    CREATE OR REPLACE SECRET onelake_storage_secret (
        TYPE AZURE,
        PROVIDER ACCESS_TOKEN,
        ACCESS_TOKEN ?
    );
""", [token])

Create a Iceberg REST catalog and attach it the DuckDB connection

In [ ]:
# Attach the Iceberg REST catalog
con.execute(f"""
    ATTACH '{read_warehouse}' AS onelake_catalog (
        TYPE ICEBERG,
        SECRET onelake_catalog_secret,
        ENDPOINT '{table_api_url}'
    );
""")

Use the DuckDB connection to run a query against our table in the lake

In [ ]:
display(con.execute("SELECT * FROM onelake_catalog.test.stations LIMIT 10").fetchdf())

## OneLake Security

In this section you'll configure column and row level security for tables in OneLake.

Read more about [OneLake Security](https://learn.microsoft.com/en-us/fabric/onelake/security/get-started-onelake-security) and how to configure [column](https://learn.microsoft.com/en-us/fabric/onelake/security/column-level-security) and [row-level](https://learn.microsoft.com/en-us/fabric/onelake/security/row-level-security) permissions.

To properly show fine grained permissions, you'll need to create a new reader user (using Entra ID). Since the process is outside the scope \
we'll switch to a live demo.  If you feel it's something you want to do, I highly recommend you try it.


## Fabric AI Functions

In this section, you'll take a look at [AI Functions](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview?tabs=pandas-pyspark%2Cpandas) -- prebuilt functions that lets you do more with your data in far less code.

Lets start by loading the libraries.

In [ ]:
from synapse.ml.spark import aifunc

In [ ]:
df = spark.sql("SELECT Description FROM readcat.dbo.fact_sale limit 10")
res = df.ai.extract(labels=["name"], input_col="Description")
res.select('name').show(truncate=False)

Lets try something more involved. Feel free to play around with the columns and prompt, as inputs to the AI function

In [ ]:
df = spark.sql("""
                select c.Customer, i.StockItem, s.Profit, s.Description
                from readcat.dbo.fact_sale as s 
                join readcat.dbo.dimension_customer as c on s.CustomerKey = c.CustomerKey
                join readcat.dbo.dimension_stock_item as i on s.StockItemKey = i.StockItemKey
                limit 10
               """)
res = df.ai.generate_response(prompt="Generate a customer purchasing habits based on the following information:", output_col="llm_response")
res.select('llm_response').limit(1).show(truncate=False)

## Fabric Data Agents

Fabric Data Agents are simple to use AI agents that allow you to ask natural language questions of your tables in OneLake. \
You simply point the agent to the tables you want to use and it does all the heavy lifting for you.

In the next section you'll programmatically create the agent, attach tables and configure it to run. \
Once ready, you'll switch back to the Fabric console to interact with it visually.

In [ ]:
%pip install fabric-data-agent-sdk -q

Import the required functions and declare the agent name and lakehouse to use

In [ ]:

from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent
)
agent_name = "iceberg_agent"
lakehouse_name = "iceberglake"

Create the data agent and configure it with basic prompt instructing it of what it is responsible for

In [ ]:
data_agent = create_data_agent(agent_name)

data_agent.update_configuration(
    instructions="You are a helpful assistant, help users with their questions",
)

ds_notes = """ \
When answering about a product sales, make sure to include the Stock Item in dimension_stock_item in the answer. 
Best selling product should be determined by sales volume, not sales amount. 
If you answer questions about quantities, make sure to include the quantity. 
"""

data_agent.add_datasource(lakehouse_name, type="lakehouse")
datasource = data_agent.get_datasources()[0]
datasource.update_configuration(instructions=ds_notes)

data_agent.publish()

Next, go back to the Fabric console, select **My Workspace** from the left menu and click **iceberg_agent**. This will open  the agent so you can interact with it. 

On the left hand-side you'll see your lakehouse **iceberglake**, click the drop-down until you see your tables under the *dbo* schema. \
Put a check in each table to select them. This adds them to the agent's context and allow it to use them to answer questions.

Next, at the prompt, ask a few questions and see what you get.

*Warning: this is using really poor sample data, which won't lead to good results so tamper your expectations*

# Thank you

This is the end of the workshop. Keep exploring!